# Indic Voice Pipeline — GPU validation on Kaggle

Training and the ASR evaluation harness already have GPU evidence under
`results/eval/`. Everything built after them — the served whisper-medium +
LoRA runner, language detection, streaming ASR, the LLM prompt/decoding
fixes, the pipeline waterfall, and the voice turn with real TTS and barge-in —
has only ever run against fakes. This notebook runs `scripts/gpu_validation.py`,
which exercises each of those against real models and writes one
`report.json` with a pass/fail per check.

**Kaggle settings (right sidebar) before running:**

- **Accelerator:** GPU T4 x2 (or P100). One GPU is used.
- **Internet:** On. Needed for the model downloads and for edge-tts, which is
  a network service.
- Optional: add a Kaggle secret named `HF_TOKEN` if the adapter or a dataset
  you point at is gated.

Wall time on a T4 is roughly 15–25 minutes, dominated by model downloads.

In [ ]:
!nvidia-smi
import os, sys
print(sys.version)
print("working dir:", os.getcwd())

## 1. Clone and install

`pyproject.toml` pins `datasets<4` (FLEURS is a script-backed dataset and
`datasets>=4` removed loading-script support). Kaggle's image may ship a newer
`datasets`; the install below downgrades it. Torch comes from the image.

In [ ]:
%cd /kaggle/working
!rm -rf indic-voice-pipeline
!git clone --depth 1 https://github.com/Vaibhav7711/indic-voice-pipeline.git
%cd indic-voice-pipeline
!pip install -q -e ".[dev]" 2>&1 | tail -3
!python scripts/preflight.py

## 2. Optional: Hugging Face token

Only needed if the adapter repo or a dataset is gated. Public repos work
without it.

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    from huggingface_hub import login
    login(token=token, add_to_git_credential=False)
    print("HF login: ok")
except Exception as exc:  # no secret configured is fine
    print(f"HF login skipped: {type(exc).__name__}")

## 3. CPU test suite

Sanity check that the checkout is healthy before spending GPU time. GPU
tests self-skip here and are run for real inside the validation script.

In [ ]:
!python -m pytest -q -rN -p no:cacheprovider 2>&1 | tail -3

## 4. Run the validation sweep

Each check is independent; a failure is recorded and the sweep continues.
The report is re-written after every check, so a timeout still leaves
partial evidence.

| Check | What it proves |
| --- | --- |
| `gpu_unit_tests` | `tests/test_asr.py`, `test_llm.py`, `test_pipeline.py` on a real GPU |
| `load_whisper_with_hub_adapter` | Hub id → local adapter → merged into the explicit runner |
| `served_model_matches_generate` | Explicit decode loop equals HF `generate()` on the *served* model and real Hindi audio |
| `asr_quality_on_clips` | Adapter is effective (WER sanity bound) at the ledger's `standard` normalization |
| `language_detection` | `language=None` detects `hi` on Hindi and `en` on English clips |
| `long_form_chunking` | > 30 s audio through chunk-and-stitch keeps the words |
| `streaming_session` | Real Whisper behind `StreamingSession`, 100 ms blocks, partials + finals |
| `llm_prompt_and_decoding` | No `<think>` in output, Hindi answer, repetition guard did not fire |
| `pipeline_waterfall` | ASR → LLM end to end, per-stage timing, peak VRAM |
| `voice_turn_real_tts` | LLM → sentence-split edge-tts streaming → playback; four latencies measured |
| `barge_in_real_tts` | Interrupt from another thread after the first real audio chunk |

In [ ]:
!python scripts/gpu_validation.py \
    --adapter Hugme6969/whisper-medium-hindi-lora \
    --out-dir results/gpu_validation

## 5. Read the report

In [ ]:
import json
from IPython.display import Markdown, display

report = json.load(open("results/gpu_validation/report.json", encoding="utf-8"))
print(json.dumps(report["context"], indent=2))
display(Markdown(open("results/gpu_validation/summary.md", encoding="utf-8").read()))

failed = [c for c in report["checks"] if c["status"] == "fail"]
for c in failed:
    print("\n=== FAILED:", c["name"], "===")
    print(c["error"])
    print(c["detail"].get("traceback", "")[-1500:])

### Latency numbers worth copying into the ledger

In [ ]:
checks = {c["name"]: c for c in report["checks"]}

def show(name, keys):
    d = checks.get(name, {}).get("detail", {})
    print(f"{name}:")
    for k in keys:
        v = d.get(k)
        print(f"  {k:40s} {v:.1f}" if isinstance(v, float) else f"  {k:40s} {v}")

show("asr_quality_on_clips", ["mean_wer", "mean_rtf"])
show("language_detection_base_model", ["mean_detection_ms", "min_probability"])
show("language_detection_adapter", ["unrestricted_wrong", "restricted_wrong"])
show("streaming_session", ["finals", "partials", "wer_vs_refs", "final_asr_ms_mean"])
show("pipeline_waterfall", ["mel_ms", "encoder_ms", "asr_decode_ms", "llm_prefill_ms",
                            "llm_decode_ms", "total_ms", "audio_to_first_llm_token_ms", "peak_vram_gib"])
show("voice_turn_real_tts", ["tts_first_chunk_ms", "final_transcript_to_first_llm_token_ms",
                             "first_llm_token_to_playback_start_ms", "response_latency_ms",
                             "first_token_is_prefill_proxy"])
print("\nLLM response:", checks.get("llm_prompt_and_decoding", {}).get("detail", {}).get("response"))
print("Turn response:", checks.get("voice_turn_real_tts", {}).get("detail", {}).get("response"))

## 6. Listen to the turn

The turn's audio went through `BufferSink` and the report keeps only chunk
counts and sizes, not the bytes. Synthesise the same response once more for a
quick listen.

In [ ]:
from IPython.display import Audio
from tts import TTSSynthesizer

resp = checks.get("voice_turn_real_tts", {}).get("detail", {}).get("response")
if resp:
    r = TTSSynthesizer(language="hi").synthesize(resp, "results/gpu_validation/turn_response.mp3")
    print(f"{r.synthesis_ms:.0f} ms, {len(r.audio_bytes)} bytes")
    display(Audio("results/gpu_validation/turn_response.mp3"))
else:
    print("no turn response recorded")

## 7. Keep the evidence

`results/gpu_validation/report.json` and `summary.md` are whitelisted in
`.gitignore`; the WAV clips and MP3 are not. Download the zip from the
notebook's Output tab, then in the repo:

```bash
unzip gpu_validation.zip -d results/
git add results/gpu_validation/report.json results/gpu_validation/summary.md
git commit -m "docs: record GPU validation of streaming, agent and pipeline"
```

and add a line to `docs/EXPERIMENTS.md` (or `docs/STREAMING.md` §7) pointing
at the report with the git SHA it was run against.

In [ ]:
!mkdir -p /kaggle/working/out
!cp results/gpu_validation/report.json results/gpu_validation/summary.md /kaggle/working/out/
!cd /kaggle/working && zip -q -r gpu_validation.zip out && ls -la gpu_validation.zip
print("Download /kaggle/working/gpu_validation.zip from the Output tab.")